In [ ]:
import sys, os, glob, subprocess
from google.colab import drive
drive.mount('/content/drive')

UTENTE = 'PaoloniCoder'
BRANCH = 'main'

!rm -rf /content/MozartAI
!git clone -b {BRANCH} https://github.com/{UTENTE}/MozartAI.git /content/MozartAI
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'gradio==6.32.0', 'pretty_midi', 'pydantic>=2.12.5,<3', 'starlette>=1.3.1,<2',
], check=True)

# Cerco i file nel repo, ovunque si trovino, e metto le loro cartelle nel percorso di Python
necessari = ['app.py', 'strumenti.py', 'pipeline.py', 'preprocessing.py', 'model.py', 'midi_convert.py', 'audio.py']
percorsi = {}
for nome in necessari:
    trovati = glob.glob(f'/content/MozartAI/**/{nome}', recursive=True)
    percorsi[nome] = trovati[0] if trovati else None
    if trovati:
        cartella = os.path.dirname(trovati[0])
        if cartella not in sys.path:
            sys.path.insert(0, cartella)

mancanti = [n for n, p in percorsi.items() if p is None]
print("Mancanti:", mancanti if mancanti else "nessuno")
assert not mancanti, "Questi file non sono nel repo (branch main): hai fatto commit e push?"
print({n: p.replace('/content/MozartAI/', '') for n, p in percorsi.items()})

In [ ]:
import importlib.util

os.environ['MOZART_MODELLO'] = '/content/drive/MyDrive/MozartAI/checkpoints_v2/best.pt'

spec = importlib.util.spec_from_file_location('mozart_app', percorsi['app.py'])
mozart_app = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mozart_app)

print("Modello:", mozart_app.precarica())
mozart_app.avvia_app(share=True, debug=True)